# Prompt Extractor v1

Notebook ini memperlihatkan bagaimana brief pengguna diubah menjadi fakta bersumber sebelum halaman dibuat. Model extractor adalah checkpoint terpisah dari generator TRAIN-025.

Corpus train memakai keluarga `train` dari V5. Development dibuat dari profil sintetis baru dengan pola prompt berbeda. Notebook tidak membuka validation lama, test, atau final. Generation website dan training tidak berjalan otomatis dari Run All.

## 1. Persiapan lingkungan

Jalankan notebook dari root `Vibe_Coding_Proyek_1` menggunakan kernel proyek Python 3.11 CUDA. Semua artefak data, metrik, dan checkpoint extractor tetap lokal serta diabaikan Git.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'src/fullpage/request_extract_contracts.py').is_file():
    raise RuntimeError('Buka notebook dari root proyek Vibe_Coding_Proyek_1')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / 'data/processed/request_extract/v1'
OUTPUT_DIR = ROOT / 'outputs/request_extract/v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Root:', ROOT)
print('Dataset tersedia:', (DATA_DIR / 'manifest.json').is_file())

## 2. Siapkan corpus terisolasi

Cell ini hanya menyusun dataset. Ia tidak membaca request V5 development, test, maupun final. Setiap contoh memuat scope extractor agar output fakta yang panjang dapat dibagi ke beberapa decode internal tanpa menambah pertanyaan untuk pengguna.

In [ ]:
from qa.prepare_request_extract_v1 import build_request_extract_dataset

if not (DATA_DIR / 'manifest.json').is_file():
    manifest = build_request_extract_dataset(ROOT, DATA_DIR.relative_to(ROOT))
else:
    manifest = json.loads((DATA_DIR / 'manifest.json').read_text(encoding='utf-8'))
print(json.dumps({
    'family_counts': manifest['family_counts'],
    'row_counts': manifest['row_counts'],
    'train_fact_family_counts': manifest['train_fact_family_counts'],
    'train_genre_family_counts': manifest['train_genre_family_counts'],
    'development_bucket_family_counts': manifest['development_bucket_family_counts'],
    'output_budgets': manifest['output_budgets'],
    'maxima': manifest['maxima'],
    'final_used': manifest['final_used'],
}, ensure_ascii=False, indent=2))

## 3. Audit bukti dan budget token

Audit memeriksa checksum, keluarga dan pola prompt yang tidak tumpang tindih, bukti teks exact, enum katalog, ukuran bucket, serta budget gabungan prefix dan target di dalam konteks 2.048 token.

In [ ]:
from qa.audit_request_extract_v1 import audit_dataset

data_audit = audit_dataset(ROOT, DATA_DIR.relative_to(ROOT))
(OUTPUT_DIR / 'data-audit.json').write_text(json.dumps(data_audit, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(data_audit, ensure_ascii=False, indent=2))

## 4. Inspeksi contoh input dan target

Contoh berikut menunjukkan satu scope dari satu keluarga train. Nilai model bukan RequestDraft dan belum dianggap benar sebelum service server membuktikan seluruh nilai terhadap pesan sumber.

In [ ]:
from src.data.normalize import read_jsonl

train_rows = list(read_jsonl(DATA_DIR / 'train.jsonl'))
example = next(row for row in train_rows if row['scenario'] == 'complete' and row['scope'] == 'business_scalars')
target = json.loads(example['target'])
print('Keluarga train:', example['family_id'])
print('Scope:', example['scope'])
print('Pesan sumber disamarkan:')
print(json.dumps([{'id': message['id'], 'char_count': len(message['text']), 'text': '[ISI BRIEF DISEMBUNYIKAN]'} for message in example['messages']], ensure_ascii=False, indent=2))
print('Label fakta disamarkan, ID sumber dipertahankan:')
print(json.dumps([{'kind': fact['kind'], 'value': '[NILAI DISEMBUNYIKAN]', 'source_message_id': fact['source_message_id']} for fact in target['fact_mentions']], ensure_ascii=False, indent=2))
print('Budget:', json.dumps(example['budget'], ensure_ascii=False))

## 5. Status training dan evaluasi

Training hanya dapat dijalankan setelah preflight storage/CUDA dan audit manifest lulus. Development tidak dipakai untuk menyetel prompt; ia hanya menilai checkpoint yang dibekukan. Bagian training dan laporan metrik ditambahkan setelah trainer serta gate implementasi tersedia. Partisi final tetap tertutup.